# 03 — Model Comparison (Qualitative)

**Tujuan**: bandingkan output top-5 dari 4 model (TF-IDF, BM25, IndoBERT, Hybrid) untuk sample queries.
Untuk rubric Model 20% — wajib show qualitative per-query analysis.

## Setup

In [ ]:
import sys
import json
from pathlib import Path

sys.path.insert(0, '../backend')

from app.indexing.bm25 import BM25Index
from app.indexing.tfidf import TFIDFIndex
from app.indexing.indobert import IndoBERTIndex
from app.indexing.hybrid import HybridIndex
from app.preprocessing import PreprocessingPipeline

## Load Indexes + Corpus

In [ ]:
INDEXES_DIR = Path('../data/indexes')
tfidf = TFIDFIndex.load(INDEXES_DIR / 'tfidf.pkl')
bm25 = BM25Index.load(INDEXES_DIR / 'bm25.pkl')
indobert = IndoBERTIndex.load(INDEXES_DIR / 'indobert')
hybrid = HybridIndex(bm25, indobert, bm25_top_k=50, alpha=0.3)

pipeline = PreprocessingPipeline()

with open('../data/processed/corpus.json', encoding='utf-8') as f:
    corpus = {item['id']: item for item in json.load(f)}

print(f'Loaded 4 indexes, {len(corpus)} docs in corpus')

## Helper: Show Top-5 Side-by-Side

In [ ]:
def show_query(q: str, top_k: int = 5):
    processed = pipeline.process(q).processed
    print(f'Query:     {q!r}')
    print(f'Processed: {processed!r}')
    print('=' * 80)
    
    for index, name in [(tfidf, 'TF-IDF'), (bm25, 'BM25'), (indobert, 'IndoBERT'), (hybrid, 'Hybrid')]:
        hits = index.query(processed, top_k=top_k)
        print(f'\n[{name}]')
        for h in hits:
            meta = corpus[h.doc_id]['metadata']
            print(f'  rank{h.rank} score={h.score:.4f}')
            print(f'    {meta["judul"]}')
            print(f'    Rp {meta["harga_per_bulan"]:,} / {meta["tipe"]} / {meta["kecamatan"]}')

## Query Analysis

### Query 1: Multi-Constraint UNILA

In [ ]:
show_query('kos putra dekat unila wifi dan ac')

### Query 2: Multi-University ITERA

In [ ]:
show_query('kos dekat itera sukarame kamar mandi dalam')

### Query 3: Fasilitas + Tipe

In [ ]:
show_query('kos campur kamar mandi dalam parkir motor')

### Query 4: Premium / Eksklusif

In [ ]:
show_query('kos eksklusif ac wifi kamar mandi dalam shower')

### Query 5: Kualitas / Keamanan

In [ ]:
show_query('kos putri aman bersih ada cctv')

## Insights per Model

- **BM25** menang di queries dengan exact-match nama kampus / fasilitas yang muncul di deskripsi
- **IndoBERT** kuat di pool-restricted (semantic mengelompokkan kos sejenis), tertekan di standard karena GT lexical-pooled
- **Hybrid** #2 di pool-restricted (MRR tertinggi) — kombinasi bernilai saat eval fair
- **TF-IDF** kompetitif di corpus kecil 227 dengan term jarang (selisih vs BM25 tidak signifikan)